# Notebook 21a — County-Level Gap Extraction for HOLC Merge
### Persistent Racial Disparities in U.S. Mortgage Approval

**Purpose:** Extract county-level racial approval gaps from raw HMDA files.
This is a lightweight extraction — reads only 3 columns from each raw file,
no full cleaning pipeline required.

**Why county-level not state-level:** State-level HOLC aggregation is too
coarse. HOLC grading was city/county-specific. County-level gaps give
a signal that state-level would dilute by a factor of 10–50x.

**Input:** `data/hmda_{year}.csv` (raw files, 2020–2024)
**Output:** `data/processed/county_gap_by_year.csv`
**Runtime:** ~45 minutes
**RAM:** ~4 GB peak (reads in chunks)


In [ ]:
"""
NOTEBOOK 21a: COUNTY-LEVEL GAP EXTRACTION
==========================================
Reads raw HMDA files and extracts county-level approval gaps.
Uses chunked reading — safe for 16GB RAM.

COLUMNS NEEDED:
  county_code        — 5-digit FIPS code (state + county)
  applicant_race_1   — race code (3=Black, 5=White)
  action_taken       — 1=approved, 3=denied
  
That is all. No LTV, no income, no loan amount.
This is a pure geography + race + outcome extraction.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_DIR  = Path('../data')
PROC_DATA_DIR = Path('../data/processed')
TABLES_DIR    = Path('../outputs/tables')
PROC_DATA_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

YEARS      = [2020, 2021, 2022, 2023, 2024]
BLACK_CODE = 3
WHITE_CODE = 5
CHUNK_SIZE = 300_000

# Columns to extract — just what we need
EXTRACT_COLS = ['county_code', 'applicant_race_1', 'action_taken', 'activity_year']

print('='*65)
print('NOTEBOOK 21a: COUNTY-LEVEL GAP EXTRACTION')
print('='*65)
print()
print('This notebook extracts county-level racial approval gaps')
print('for merging with HOLC redlining shapefiles in NB21b.')
print()

# Check raw files exist
print('Checking raw files...')
available_years = []
for yr in YEARS:
    for name in [f'hmda_{yr}.csv',
                 f'hmda_{yr}_nationwide_all-records_labels.csv',
                 f'{yr}_public_lar_csv.csv']:
        if (RAW_DATA_DIR / name).exists():
            print(f'  {yr}: found {name}')
            available_years.append((yr, RAW_DATA_DIR / name))
            break
    else:
        print(f'  {yr}: NOT FOUND — skip')

print(f'\nWill process {len(available_years)} years')


In [ ]:
# =============================================================================
# EXTRACTION FUNCTION — vectorized (safe for 16GB RAM, fast on i7-13750HX)
# =============================================================================

def extract_county_gaps(year, filepath):
    """
    Extract county-level approval counts from one raw HMDA file.
    VECTORIZED — uses groupby, not iterrows. Safe on 16GB RAM.
    Runtime: ~8 minutes per year on i7-13750HX.
    """
    print(f'\nProcessing {year}...')

    chunks_read   = 0
    yearly_chunks = []

    reader = pd.read_csv(
        filepath,
        chunksize=CHUNK_SIZE,
        usecols=lambda c: c in EXTRACT_COLS,
        engine='python',
        on_bad_lines='skip',
        dtype=str
    )

    for chunk in reader:
        # Numeric conversion
        chunk['applicant_race_1'] = pd.to_numeric(
            chunk['applicant_race_1'], errors='coerce')
        chunk['action_taken'] = pd.to_numeric(
            chunk['action_taken'], errors='coerce')

        # Keep only Black/White, originated/denied, non-null county
        chunk = chunk[
            chunk['applicant_race_1'].isin([BLACK_CODE, WHITE_CODE]) &
            chunk['action_taken'].isin([1, 3]) &
            chunk['county_code'].notna()
        ].copy()

        if len(chunk) == 0:
            chunks_read += 1
            continue

        # Vectorized derivations
        chunk['approved'] = (chunk['action_taken'] == 1).astype(int)
        chunk['black']    = (chunk['applicant_race_1'] == BLACK_CODE).astype(int)
        chunk['county_fips'] = chunk['county_code'].astype(str).str.strip().str.zfill(5)

        yearly_chunks.append(
            chunk[['county_fips', 'black', 'approved']].copy()
        )
        chunks_read += 1

        if chunks_read % 20 == 0:
            total_rows = sum(len(c) for c in yearly_chunks)
            print(f'  chunk {chunks_read}: {total_rows:,} rows kept so far')

    if not yearly_chunks:
        print(f'  No data found for {year}')
        return pd.DataFrame()

    # Combine all chunks and aggregate with groupby — VECTORIZED
    df_year = pd.concat(yearly_chunks, ignore_index=True)

    # Compute county-level stats in one groupby pass
    gb = df_year.groupby(['county_fips', 'black'])['approved'].agg(
        ['sum', 'count']).reset_index()
    gb.columns = ['county_fips', 'black', 'approved_sum', 'n']

    black_df = gb[gb['black']==1].rename(
        columns={'approved_sum':'ba', 'n':'bt'}).drop('black', axis=1)
    white_df = gb[gb['black']==0].rename(
        columns={'approved_sum':'wa', 'n':'wt'}).drop('black', axis=1)

    merged = black_df.merge(white_df, on='county_fips', how='inner')
    merged = merged[(merged['bt'] >= 20) & (merged['wt'] >= 20)].copy()

    merged['b_rate'] = merged['ba'] / merged['bt']
    merged['w_rate'] = merged['wa'] / merged['wt']
    merged['gap_pp'] = (merged['w_rate'] - merged['b_rate']) * 100

    result = merged.rename(columns={'bt':'black_n','wt':'white_n'})[[
        'county_fips', 'black_n', 'white_n', 'gap_pp'
    ]].copy()
    result['year'] = year
    result['black_approval_rate'] = (merged['b_rate'] * 100).round(4).values
    result['white_approval_rate'] = (merged['w_rate'] * 100).round(4).values
    result['gap_pp'] = result['gap_pp'].round(4)

    print(f'  Done: {len(result):,} counties with ≥20 Black and ≥20 White apps')
    print(f'  Mean gap: {result["gap_pp"].mean():.2f}pp  '
          f'Range: {result["gap_pp"].min():.2f} to {result["gap_pp"].max():.2f}pp')
    return result


In [ ]:
# =============================================================================
# RUN EXTRACTION FOR ALL AVAILABLE YEARS
# =============================================================================

all_dfs = []

for year, filepath in available_years:
    df_yr = extract_county_gaps(year, filepath)
    all_dfs.append(df_yr)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)

    # Pooled county gap (average across years)
    county_pooled = df_all.groupby('county_fips').agg(
        black_n    = ('black_n', 'sum'),
        white_n    = ('white_n', 'sum'),
        gap_pp_mean= ('gap_pp', 'mean'),
        gap_pp_sd  = ('gap_pp', 'std'),
        n_years    = ('year', 'count'),
    ).reset_index()

    county_pooled['state_fips'] = county_pooled['county_fips'].str[:2]

    print('\n' + '='*65)
    print('COUNTY GAP SUMMARY (POOLED 2020-2024)')
    print('='*65)
    print(f'Counties with data: {len(county_pooled):,}')
    print(f'Mean gap:           {county_pooled["gap_pp_mean"].mean():.2f} pp')
    print(f'Median gap:         {county_pooled["gap_pp_mean"].median():.2f} pp')
    print(f'SD gap:             {county_pooled["gap_pp_mean"].std():.2f} pp')
    print(f'Range:              {county_pooled["gap_pp_mean"].min():.2f} to '
          f'{county_pooled["gap_pp_mean"].max():.2f} pp')

    # Save both year-by-year and pooled
    out_yearly = PROC_DATA_DIR / 'county_gap_by_year.csv'
    out_pooled = PROC_DATA_DIR / 'county_gap_pooled.csv'
    df_all.to_csv(out_yearly, index=False)
    county_pooled.to_csv(out_pooled, index=False)

    print(f'\nSaved: {out_yearly}')
    print(f'Saved: {out_pooled}')
    print('\n✅ NB21a COMPLETE — ready for HOLC merge in NB21b')
else:
    print('No data extracted — check raw file paths')
